<a href="https://colab.research.google.com/github/veerannagariharshith-prog/city-energy-consumption-analysis2/blob/main/City_Energy_Consumption_Analysis%5B1%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# City Energy Consumption Analysis & Prediction System

End-to-end project: synthetic data generation, cleaning, analysis, visualization, next-day prediction, MAE evaluation, and interactive prediction.

In [13]:
!wget -q https://raw.githubusercontent.com/veerannagirharshith-prog/city-energy-consumption-analysis2/main/city_energy_analysis.py

In [14]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Simplify path handling to directly add the current working directory to sys.path
# This ensures the downloaded 'city_energy_analysis.py' is always found.
sys.path.insert(0, os.getcwd())

# Changed from specific imports to general module import to avoid ImportError
# if the specific names are not directly at the top level, as was seen in GR1vjYiGe5Sc.
# The functions will now need to be called as city_energy_analysis.generate_dataset()
import city_energy_analysis

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import sys
import os

!wget -q -O city_energy_analysis.py https://raw.githubusercontent.com/veerannagirharshith-prog/city-energy-consumption-analysis2/main/city_energy_analysis.py

sys.path.insert(0, os.getcwd())

# Changed from specific imports to general module import to avoid ImportError
# if the specific names are not directly at the top level.
# The functions will now need to be called as city_energy_analysis.generate_dataset()
import city_energy_analysis

print("✅ city_energy_analysis.py loaded successfully!")

## 1. Generate and clean the dataset

In [ ]:
df = city_energy_analysis.clean_data(city_energy_analysis.generate_dataset(days=365))
print(f"Rows: {len(df):,}")
print(f"Zones: {df['ZoneID'].nunique()}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
display(df.head())

## 2. Data quality checks

In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("Missing"))
print("Duplicate rows:", df.duplicated().sum())
display(df.describe().round(2))

## 3. Average consumption per month and per zone

In [ ]:
df["Month"] = df["Date"].dt.to_period("M").astype(str)
monthly = df.groupby("Month", as_index=False)["EnergyConsumption"].mean()
zone_summary = df.groupby("ZoneID")["EnergyConsumption"].agg(["mean", "min", "max"]).round(2)
display(monthly)
display(zone_summary)

## 4. Required visualization — monthly trend

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(monthly["Month"], monthly["EnergyConsumption"], marker="o")
plt.xticks(rotation=45)
plt.title("Average Daily Energy Consumption by Month")
plt.xlabel("Month")
plt.ylabel("Average Consumption (kWh)")
plt.tight_layout()
plt.show()

## 5. Required visualization — correlation heatmap

In [ ]:
corr_cols = ["AvgTemperature", "Humidity", "SpecialEvent", "EnergyConsumption"]
plt.figure(figsize=(7, 5))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Between Weather, Events, and Energy Consumption")
plt.tight_layout()
plt.show()

## 6. Required visualization — event vs non-event

In [ ]:
event_means = (
    df.assign(EventStatus=df["SpecialEvent"].map({0: "No Event", 1: "Event"}))
      .groupby("EventStatus")["EnergyConsumption"]
      .mean()
      .reindex(["No Event", "Event"])
)
plt.figure(figsize=(7, 5))
plt.bar(event_means.index, event_means.values)
plt.title("Average Energy Consumption: Event vs Non-Event")
plt.xlabel("Day Type")
plt.ylabel("Average Consumption (kWh)")
plt.tight_layout()
plt.show()
display(event_means.to_frame("Average kWh").round(2))

## 7. Train and evaluate the next-day Random Forest

In [ ]:
model_data = add_next_day_target(df)
model, features, train, test, predictions, mae = train_model(df)

print(f"Training rows: {len(train):,}")
print(f"Testing rows: {len(test):,}")
print(f"Test MAE: {mae:,.2f} kWh")

results = test[["Date", "ZoneID", "TargetNextDayConsumption"]].copy()
results["Predicted"] = predictions
results["AbsoluteError"] = (results["TargetNextDayConsumption"] - results["Predicted"]).abs()
display(results.head(10).round(2))

## 8. Feature importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
display(importance.to_frame("Importance").round(4))
plt.figure(figsize=(7, 4))
plt.barh(importance.index, importance.values)
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 9. Interactive prediction example

In [ ]:
zone_id = 3
tomorrow_temperature = 31.0
tomorrow_humidity = 68.0
event_indicator = 0

prediction = predict_tomorrow(
    model, zone_id, tomorrow_temperature, tomorrow_humidity, event_indicator
)
print(f"Predicted next-day consumption for Zone {zone_id}: {prediction:,.2f} kWh")

## 10. Key findings

- Dataset size: **1,825 readings** across **5 zones**.
- Seasonal temperature variation produces monthly demand variation.
- Zones have distinct baseline demand levels.
- Temperature, humidity, and events influence simulated consumption.
- Event days generally have higher simulated demand.
- Random Forest test MAE on this generated dataset: **263.78 kWh**.

The dataset is synthetic, so these findings demonstrate the project workflow rather than real-world electricity demand.